# Ejercicios — Tipos de Ordenes y Matching Behavior

Estos ejercicios continuan directamente desde L4: usamos los mismos 500 snapshots de `btc_lob_snapshots.csv`.

**Tiers:**
- **Nucleo (1–5):** obligatorio en clase — market buy, slippage, market sell, limit order
- **Si vamos bien (6–7):** IOC y FOK — si queda tiempo
- **Bonus / casa (8–10):** coste comparativo, secuencia, modelo maker/taker fee

Cada ejercicio tiene un validador que te dice si lo has hecho bien. Busca el mensaje `Bien:` al final.

---

## Ejercicio 0 — El libro que vamos a atacar (motivacional)

Sin escribir codigo: observa el snapshot 0 del LOB de L4. Este es el libro con el que vas a interactuar en los ejercicios.

Responde mentalmente antes de ejecutar la celda:
1. Si envias una **market buy de 1 BTC**, ¿cuantos niveles del ask consumiras?
2. Si envias una **market buy de 5 BTC**, ¿a que precio medio ejecutarias? ¿Cuanto slippage?
3. Si colocas una **limit bid a $99.960**, ¿en que posicion del bid side queda?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import copy
import uuid

df = pd.read_csv("../../04-market-microstructure-btc/data/btc_lob_snapshots.csv")
row = df.iloc[0]

# Visualizar el snapshot 0
bids = [(row[f"bid_price_{i}"], row[f"bid_size_{i}"]) for i in range(1, 11)]
asks = [(row[f"ask_price_{i}"], row[f"ask_size_{i}"]) for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 5))
bid_prices, bid_sizes = zip(*bids)
ask_prices, ask_sizes = zip(*asks)
ax.barh(bid_prices, bid_sizes, height=3, color="#4ade80", alpha=0.8, label="Bids")
ax.barh(ask_prices, ask_sizes, height=3, color="#f87171", alpha=0.8, label="Asks")
ax.axhline(y=(bid_prices[0] + ask_prices[0]) / 2, color="#22d3ee", linestyle="--", linewidth=1, label="Mid")
ax.set_xlabel("Volumen (BTC)")
ax.set_ylabel("Precio (USD)")
ax.set_title(f"Snapshot 0 — Best bid ${bid_prices[0]:,.2f} | Best ask ${ask_prices[0]:,.2f} | Spread ${ask_prices[0]-bid_prices[0]:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

print("\nVolumen total en asks:")
for p, s in asks[:5]:
    print(f"  ${p:,.2f} x {s:.4f} BTC")
print(f"  ... (acumulado top 5: {sum(s for _, s in asks[:5]):.4f} BTC)")

---

## Ejercicio 1 — Cargar el LOB y construir el MatchingEngine

Carga el CSV de L4 y construye un `MatchingEngine` con el snapshot 0.

El motor debe:
- Tener una lista `self.bids` y `self.asks` de dicts con claves `price`, `size`, `id`
- Exponer metodos `best_bid()`, `best_ask()`, `spread()`
- Tener un constructor de clase `from_row(cls, row)` que lea las columnas del CSV

**Pista:** los nombres de columna del CSV son `bid_price_1`, `bid_size_1`, ..., `bid_price_10`, `bid_size_10` (igual para ask).

In [ ]:
class MatchingEngine:
    def __init__(self, bids, asks):
        self.bids = copy.deepcopy(bids)
        self.asks = copy.deepcopy(asks)

    @classmethod
    def from_row(cls, row):
        # TODO: construir bids y asks como listas de dicts {price, size, id}
        bids = []
        asks = []
        return cls(bids, asks)

    def best_bid(self):
        # TODO
        pass

    def best_ask(self):
        # TODO
        pass

    def spread(self):
        # TODO
        pass


engine = MatchingEngine.from_row(row)

In [ ]:
# Validador ejercicio 1
try:
    e = MatchingEngine.from_row(row)
    assert len(e.bids) == 10, f"Se esperaban 10 niveles bid, hay {len(e.bids)}"
    assert len(e.asks) == 10, f"Se esperaban 10 niveles ask, hay {len(e.asks)}"
    assert "price" in e.bids[0] and "size" in e.bids[0], "Falta clave 'price' o 'size' en bids[0]"
    bb = e.best_bid()
    ba = e.best_ask()
    assert bb is not None and ba is not None, "best_bid() o best_ask() devuelven None"
    assert bb < ba, f"best_bid ({bb}) debe ser menor que best_ask ({ba})"
    sp = e.spread()
    assert abs(sp - (ba - bb)) < 0.01, f"spread() incorrecto: {sp} vs {ba - bb}"
    print(f"Bien: MatchingEngine construido — bid=${bb:,.2f}, ask=${ba:,.2f}, spread=${sp:.2f}")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 1</summary>

```python
class MatchingEngine:
    def __init__(self, bids, asks):
        self.bids = copy.deepcopy(bids)
        self.asks = copy.deepcopy(asks)

    @classmethod
    def from_row(cls, row):
        bids = [{"price": row[f"bid_price_{i}"], "size": row[f"bid_size_{i}"], "id": f"b{i}"}
                for i in range(1, 11)]
        asks = [{"price": row[f"ask_price_{i}"], "size": row[f"ask_size_{i}"], "id": f"a{i}"}
                for i in range(1, 11)]
        return cls(bids, asks)

    def best_bid(self):
        return self.bids[0]["price"] if self.bids else None

    def best_ask(self):
        return self.asks[0]["price"] if self.asks else None

    def spread(self):
        if self.bids and self.asks:
            return self.asks[0]["price"] - self.bids[0]["price"]
        return None
```
</details>

---

## Ejercicio 2 — Implementar `market_buy`

Implementa la funcion `market_buy(engine, size)` que:
1. Camina los niveles del ask side de menor a mayor precio
2. Registra cada fill con `{price, size}`
3. Actualiza el libro (reduce/elimina los niveles consumidos)
4. Devuelve `{fills, total_filled, avg_price, slippage, residual}`

El **slippage** es la diferencia entre el precio medio de ejecucion y el best ask original.

**Pista:** usa `min(remaining, level["size"])` para calcular el fill en cada nivel.

In [ ]:
def market_buy(engine, size):
    orig_ask = engine.best_ask()
    fills, remaining, cost = [], size, 0.0

    for level in engine.asks:
        if remaining <= 0:
            break
        # TODO: calcular filled, anotar fill, actualizar level y remaining
        pass

    # TODO: eliminar niveles vacios de engine.asks

    total_filled = round(size - remaining, 6)
    avg_price = cost / total_filled if total_filled > 0 else 0
    slippage = avg_price - orig_ask if avg_price > 0 else 0

    return {
        "fills": fills,
        "total_filled": total_filled,
        "avg_price": round(avg_price, 4),
        "slippage": round(slippage, 4),
        "residual": remaining,
    }


engine = MatchingEngine.from_row(row)
result = market_buy(engine, 0.5)
print(result)

In [ ]:
# Validador ejercicio 2
try:
    e = MatchingEngine.from_row(row)
    r = market_buy(e, 0.5)
    assert abs(r["total_filled"] - 0.5) < 0.001, f"total_filled esperado 0.5, obtenido {r['total_filled']}"
    assert r["residual"] == 0, f"residual debe ser 0 para una orden que cabe en el libro"
    assert len(r["fills"]) >= 1, "fills debe tener al menos una entrada"
    assert r["avg_price"] > 0, "avg_price debe ser positivo"
    assert r["slippage"] >= 0, "slippage de un market buy debe ser >= 0"
    # Verificar que el libro se actualizo
    total_ask_after = sum(l["size"] for l in e.asks)
    total_ask_before = sum(row[f"ask_size_{i}"] for i in range(1, 11))
    assert abs(total_ask_before - total_ask_after - 0.5) < 0.01, "El libro no se actualizo correctamente"
    print(f"Bien: market_buy 0.5 BTC — filled={r['total_filled']}, avg=${r['avg_price']:,.2f}, slippage=${r['slippage']:.4f}")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 2</summary>

```python
def market_buy(engine, size):
    orig_ask = engine.best_ask()
    fills, remaining, cost = [], size, 0.0

    for level in engine.asks:
        if remaining <= 0:
            break
        filled = min(remaining, level["size"])
        fills.append({"price": level["price"], "size": round(filled, 6)})
        cost += filled * level["price"]
        level["size"] = round(level["size"] - filled, 6)
        remaining = round(remaining - filled, 6)

    engine.asks = [l for l in engine.asks if l["size"] > 0.0001]

    total_filled = round(size - remaining, 6)
    avg_price = cost / total_filled if total_filled > 0 else 0
    slippage = avg_price - orig_ask if avg_price > 0 else 0

    return {
        "fills": fills,
        "total_filled": total_filled,
        "avg_price": round(avg_price, 4),
        "slippage": round(slippage, 4),
        "residual": remaining,
    }
```
</details>

---

## Ejercicio 3 — Slippage en funcion del tamano

Usa `market_buy` para calcular el slippage para distintos tamanos: 0.5, 1, 2, 5 y 8 BTC.

Muestra los resultados en una tabla y en un grafico de linea (tamano vs slippage).

**Atencion:** debes recrear el motor para cada tamano — cada `market_buy` modifica el libro.

In [ ]:
sizes = [0.5, 1.0, 2.0, 5.0, 8.0]
slippages = []
avg_prices = []

for s in sizes:
    # TODO: crear un motor fresco, ejecutar market_buy(engine, s), guardar slippage y avg_price
    pass

# TODO: tabla y grafico
print("Tamano (BTC) | Precio medio | Slippage ($)")
print("-" * 45)

In [ ]:
# Validador ejercicio 3
try:
    assert len(slippages) == 5, f"Faltan resultados: hay {len(slippages)}, se esperaban 5"
    assert slippages[0] >= 0, "Slippage debe ser >= 0"
    assert slippages[-1] >= slippages[0], "El slippage debe crecer (o igual) con el tamano"
    assert all(s >= 0 for s in slippages), "Todos los slippages deben ser >= 0"
    print(f"Bien: slippages calculados — {[round(s, 4) for s in slippages]}")
    print(f"El slippage a 8 BTC es {slippages[-1]/slippages[0]:.1f}x mayor que a 0.5 BTC")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 3</summary>

```python
sizes = [0.5, 1.0, 2.0, 5.0, 8.0]
slippages = []
avg_prices = []

for s in sizes:
    e = MatchingEngine.from_row(row)
    r = market_buy(e, s)
    slippages.append(r["slippage"])
    avg_prices.append(r["avg_price"])

print("Tamano (BTC) | Precio medio   | Slippage ($)")
print("-" * 45)
for s, ap, sl in zip(sizes, avg_prices, slippages):
    print(f"  {s:<12} ${ap:,.2f}       ${sl:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, slippages, marker="o", color="#f87171")
ax.set_xlabel("Tamano (BTC)")
ax.set_ylabel("Slippage ($)")
ax.set_title("Slippage vs tamano de la orden")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

---

## Ejercicio 4 — Implementar `market_sell`

Implementa `market_sell(engine, size)` — la version simetrica de `market_buy`:
- Camina los **bids** de mayor a menor precio
- Calcula `slippage = best_bid_original - avg_price` (cuanto menos cobras respecto al best bid)

**Pista:** la estructura es identica a `market_buy` pero iterando `engine.bids`.

In [ ]:
def market_sell(engine, size):
    # TODO
    pass


engine = MatchingEngine.from_row(row)
r = market_sell(engine, 0.5)
print(r)

In [ ]:
# Validador ejercicio 4
try:
    e = MatchingEngine.from_row(row)
    r = market_sell(e, 0.5)
    assert r is not None, "market_sell devuelve None — falta implementacion"
    assert abs(r["total_filled"] - 0.5) < 0.001, f"total_filled esperado 0.5, obtenido {r['total_filled']}"
    assert r["slippage"] >= 0, "slippage de market_sell debe ser >= 0 (cobras menos que el best bid)"
    # El avg_price debe ser <= best_bid
    bb_orig = row["bid_price_1"]
    assert r["avg_price"] <= bb_orig + 0.01, f"avg_price ({r['avg_price']}) debe ser <= best_bid ({bb_orig})"
    # Verificar que el libro se actualizo
    total_bid_after = sum(l["size"] for l in e.bids)
    total_bid_before = sum(row[f"bid_size_{i}"] for i in range(1, 11))
    assert abs(total_bid_before - total_bid_after - 0.5) < 0.01, "El libro bid no se actualizo correctamente"
    print(f"Bien: market_sell 0.5 BTC — filled={r['total_filled']}, avg=${r['avg_price']:,.2f}, slippage=${r['slippage']:.4f}")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 4</summary>

```python
def market_sell(engine, size):
    orig_bid = engine.best_bid()
    fills, remaining, revenue = [], size, 0.0

    for level in engine.bids:
        if remaining <= 0:
            break
        filled = min(remaining, level["size"])
        fills.append({"price": level["price"], "size": round(filled, 6)})
        revenue += filled * level["price"]
        level["size"] = round(level["size"] - filled, 6)
        remaining = round(remaining - filled, 6)

    engine.bids = [l for l in engine.bids if l["size"] > 0.0001]

    total_filled = round(size - remaining, 6)
    avg_price = revenue / total_filled if total_filled > 0 else 0
    slippage = orig_bid - avg_price if avg_price > 0 else 0

    return {
        "fills": fills,
        "total_filled": total_filled,
        "avg_price": round(avg_price, 4),
        "slippage": round(slippage, 4),
        "residual": remaining,
    }
```
</details>

---

## Ejercicio 5 — Colocar una limit order

Implementa `place_limit(engine, side, price, size)` que:
1. Detecta si la orden **cruza el spread** (bid con precio >= best ask, o ask con precio <= best bid)
   - Si cruza: ejecuta como market order
2. Si no cruza: inserta la orden en la posicion correcta del libro
   - Bids: ordenados de mayor a menor precio
   - Asks: ordenados de menor a mayor precio

Devuelve un dict con `type` (`"LIMIT-RESTING"` o `"LIMIT-CROSS"`), `id`, `side`, `price`, `size`.

In [ ]:
def place_limit(engine, side, price, size):
    order_id = str(uuid.uuid4())[:8]

    # TODO: detectar cruce de spread y ejecutar como market si cruza

    # TODO: insertar en posicion correcta del libro

    pass


# Pruebas
e1 = MatchingEngine.from_row(row)
r1 = place_limit(e1, "bid", 99950.0, 0.5)  # por debajo del spread — debe quedar en cola
print("Limit bid por debajo del spread:", r1)

e2 = MatchingEngine.from_row(row)
r2 = place_limit(e2, "bid", 100030.0, 0.3)  # por encima del ask — debe ejecutar
print("Limit bid que cruza el spread:", r2)

In [ ]:
# Validador ejercicio 5
try:
    # Caso 1: limit resting
    e = MatchingEngine.from_row(row)
    r = place_limit(e, "bid", 99950.0, 0.5)
    assert r is not None, "place_limit devuelve None"
    assert r.get("type") == "LIMIT-RESTING", f"tipo esperado LIMIT-RESTING, obtenido {r.get('type')}"
    # Verificar que la orden esta en el libro
    found = any(b.get("price") == 99950.0 for b in e.bids)
    assert found, "La orden limit no se encontro en engine.bids"

    # Caso 2: limit que cruza
    e2 = MatchingEngine.from_row(row)
    r2 = place_limit(e2, "bid", 100030.0, 0.3)
    assert r2.get("type") == "LIMIT-CROSS", f"tipo esperado LIMIT-CROSS, obtenido {r2.get('type')}"
    assert r2.get("total_filled", 0) > 0, "La orden que cruza el spread debe tener fills"

    print(f"Bien: limit resting en ${99950:.2f} y limit-cross ejecutado con {r2['total_filled']:.4f} BTC")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 5</summary>

```python
def place_limit(engine, side, price, size):
    order_id = str(uuid.uuid4())[:8]

    if side == "bid" and engine.asks and price >= engine.asks[0]["price"]:
        return {**market_buy(engine, size), "type": "LIMIT-CROSS", "id": order_id}

    if side == "ask" and engine.bids and price <= engine.bids[0]["price"]:
        return {**market_sell(engine, size), "type": "LIMIT-CROSS", "id": order_id}

    new_level = {"price": price, "size": size, "id": order_id, "is_user": True}

    if side == "bid":
        idx = next((i for i, b in enumerate(engine.bids) if b["price"] <= price), len(engine.bids))
        engine.bids.insert(idx, new_level)
    else:
        idx = next((i for i, a in enumerate(engine.asks) if a["price"] >= price), len(engine.asks))
        engine.asks.insert(idx, new_level)

    return {"type": "LIMIT-RESTING", "id": order_id, "side": side, "price": price, "size": size}
```
</details>

---

## Ejercicio 6 — IOC (Immediate or Cancel) *(Si vamos bien)*

Implementa `ioc_buy(engine, size)` que:
- Ejecuta todo lo que pueda del ask side
- Si queda residuo sin ejecutar, lo **cancela** (no se anade al libro)
- Devuelve el resultado de `market_buy` mas `"type": "IOC"` y `"cancelled": residual`

**Pista:** `market_buy` ya devuelve el residual. Solo necesitas envolverlo.

In [ ]:
def ioc_buy(engine, size):
    # TODO
    pass


engine = MatchingEngine.from_row(row)
total_ask_vol = sum(l["size"] for l in engine.asks)
print(f"Volumen total en asks: {total_ask_vol:.4f} BTC")

engine2 = MatchingEngine.from_row(row)
r = ioc_buy(engine2, total_ask_vol + 5.0)  # mas de lo disponible
print(r)

In [ ]:
# Validador ejercicio 6
try:
    e = MatchingEngine.from_row(row)
    total_avail = sum(l["size"] for l in e.asks)
    r = ioc_buy(e, total_avail + 5.0)  # pedir mas de lo disponible
    assert r is not None, "ioc_buy devuelve None"
    assert r.get("type") == "IOC", f"tipo esperado IOC, obtenido {r.get('type')}"
    assert r.get("cancelled", 0) > 0, "Con orden mayor que la liquidez, cancelled debe ser > 0"
    assert abs(r["total_filled"] - total_avail) < 0.01, (
        f"total_filled ({r['total_filled']:.4f}) debe ser aprox igual a la liquidez disponible ({total_avail:.4f})"
    )
    # Libro debe estar vacio de asks despues
    assert len(e.asks) == 0, "Asks deben estar vacios despues de consumir toda la liquidez"
    print(f"Bien: IOC buy — filled={r['total_filled']:.4f}, cancelled={r['cancelled']:.4f} BTC")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 6</summary>

```python
def ioc_buy(engine, size):
    result = market_buy(engine, size)
    return {**result, "type": "IOC", "cancelled": result["residual"]}
```
</details>

---

## Ejercicio 7 — FOK (Fill or Kill) *(Si vamos bien)*

Implementa `fok_buy(engine, size)` que:
1. **Antes** de ejecutar, verifica si hay liquidez suficiente en el ask side
2. Si no hay suficiente: devuelve `{"type": "FOK", "killed": True, ...}` sin modificar el libro
3. Si hay suficiente: ejecuta como `market_buy` y devuelve el resultado con `"killed": False`

**Pista:** `sum(l["size"] for l in engine.asks)` da el volumen total disponible.

In [ ]:
def fok_buy(engine, size):
    # TODO
    pass


# Prueba 1: FOK que ejecuta
e1 = MatchingEngine.from_row(row)
print("FOK 2.0 BTC:", fok_buy(e1, 2.0))

# Prueba 2: FOK que se mata
e2 = MatchingEngine.from_row(row)
print("FOK 20.0 BTC:", fok_buy(e2, 20.0))
print("Libro intacto despues del FOK killed:", repr(e2))

In [ ]:
# Validador ejercicio 7
try:
    # Caso 1: FOK que ejecuta
    e = MatchingEngine.from_row(row)
    r_ok = fok_buy(e, 2.0)
    assert r_ok is not None, "fok_buy devuelve None"
    assert r_ok.get("type") == "FOK", f"tipo esperado FOK, obtenido {r_ok.get('type')}"
    assert r_ok.get("killed") == False, "FOK de 2 BTC no debe ser killed (hay liquidez)"
    assert abs(r_ok["total_filled"] - 2.0) < 0.01, "total_filled debe ser ~2.0 BTC"

    # Caso 2: FOK que se mata
    e2 = MatchingEngine.from_row(row)
    asks_before = copy.deepcopy(e2.asks)
    r_kill = fok_buy(e2, 20.0)
    assert r_kill.get("killed") == True, "FOK de 20 BTC debe ser killed"
    # Libro NO debe haber cambiado
    asks_after_sizes = [l["size"] for l in e2.asks]
    asks_before_sizes = [l["size"] for l in asks_before]
    assert asks_after_sizes == asks_before_sizes, "El libro se modifico a pesar del FOK killed"

    print(f"Bien: FOK ejecuta 2 BTC (killed=False) y mata 20 BTC (killed=True, libro intacto)")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 7</summary>

```python
def fok_buy(engine, size):
    available = sum(l["size"] for l in engine.asks)
    if available < size - 0.0001:
        return {
            "type": "FOK",
            "killed": True,
            "reason": f"liquidez insuficiente: {available:.4f} BTC < {size} BTC",
            "fills": [],
            "total_filled": 0,
        }
    result = market_buy(engine, size)
    return {**result, "type": "FOK", "killed": False}
```
</details>

---

## Ejercicio 8 — Comparar coste: market vs limit *(Bonus / casa)*

Para una compra de 1 BTC, calcula:
- **Coste market:** precio medio de ejecucion del `market_buy`
- **Coste limit al mid:** precio = `(best_bid + best_ask) / 2`
- **Ahorro potencial:** diferencia entre ambos (si la limit ejecuta)

Repite el calculo para los 10 primeros snapshots del CSV y muestra el ahorro medio.

**Reflexion:** ¿Merece la pena usar limit orders siempre?

In [ ]:
ahorros = []

for i in range(10):
    row_i = df.iloc[i]
    # TODO: calcular ahorro de limit vs market para 1 BTC en el snapshot i
    pass

print(f"Ahorro medio limit vs market (10 snapshots, 1 BTC): ${sum(ahorros)/len(ahorros):.2f}")

<details>
<summary>Solucion ejercicio 8</summary>

```python
ahorros = []

for i in range(10):
    row_i = df.iloc[i]
    e = MatchingEngine.from_row(row_i)
    mid = (e.best_bid() + e.best_ask()) / 2
    r = market_buy(e, 1.0)
    ahorro = r["avg_price"] - mid  # cuanto pagas de mas con market vs limit al mid
    ahorros.append(ahorro)

print(f"Ahorro medio limit vs market (10 snapshots, 1 BTC): ${sum(ahorros)/len(ahorros):.2f}")
print(f"Rango: ${min(ahorros):.2f} — ${max(ahorros):.2f}")
print("Nota: este es el ahorro *si la limit ejecuta*. Si no ejecuta, el coste puede ser mayor.")
```
</details>

---

## Ejercicio 9 — Simular una secuencia de ordenes *(Bonus / casa)*

Implementa `simulate_sequence(engine, orders)` que procese una lista de ordenes en orden:

```python
orders = [
    {"type": "market_buy",  "size": 0.5},
    {"type": "limit_bid",   "size": 0.3, "price": 99950.0},
    {"type": "market_sell", "size": 0.5},
    {"type": "ioc_buy",     "size": 1.5},
    {"type": "fok_buy",     "size": 20.0},
]
```

Devuelve una lista con el resumen de cada ejecucion.

In [ ]:
def simulate_sequence(engine, orders):
    results = []
    for order in orders:
        # TODO: despachar segun order["type"]
        pass
    return results


engine_seq = MatchingEngine.from_row(row)
orders = [
    {"type": "market_buy",  "size": 0.5},
    {"type": "limit_bid",   "size": 0.3, "price": 99950.0},
    {"type": "market_sell", "size": 0.5},
    {"type": "ioc_buy",     "size": 1.5},
    {"type": "fok_buy",     "size": 20.0},
]

for i, r in enumerate(simulate_sequence(engine_seq, orders)):
    print(f"{i+1}. {r}")

In [ ]:
# Validador ejercicio 9
try:
    e = MatchingEngine.from_row(row)
    test_orders = [
        {"type": "market_buy",  "size": 0.5},
        {"type": "limit_bid",   "size": 0.3, "price": 99950.0},
        {"type": "market_sell", "size": 0.5},
        {"type": "ioc_buy",     "size": 1.5},
        {"type": "fok_buy",     "size": 20.0},
    ]
    results = simulate_sequence(e, test_orders)
    assert results is not None and len(results) == 5, f"Se esperaban 5 resultados, hay {len(results) if results else 0}"
    # El ultimo debe ser FOK killed (20 BTC > liquidez disponible despues de las ordenes anteriores)
    last = results[-1]
    assert last.get("killed") == True, f"El ultimo FOK deberia ser killed, obtenido: {last}"
    # El segundo deberia ser LIMIT-RESTING
    second = results[1]
    assert second.get("type") == "LIMIT-RESTING", f"La limit bid deberia ser LIMIT-RESTING, obtenido: {second.get('type')}"
    print(f"Bien: secuencia de {len(results)} ordenes procesada correctamente")
    print(f"  Orden 1 (market_buy): filled={results[0].get('total_filled', 0):.4f} BTC")
    print(f"  Orden 5 (fok_buy):    killed={last['killed']}")
except AssertionError as ex:
    print(f"Revisa: {ex}")
except Exception as ex:
    print(f"Error: {ex}")

<details>
<summary>Solucion ejercicio 9</summary>

```python
def simulate_sequence(engine, orders):
    results = []
    for order in orders:
        t, size = order["type"], order["size"]
        if t == "market_buy":
            r = market_buy(engine, size)
        elif t == "market_sell":
            r = market_sell(engine, size)
        elif t == "limit_bid":
            r = place_limit(engine, "bid", order["price"], size)
        elif t == "limit_ask":
            r = place_limit(engine, "ask", order["price"], size)
        elif t == "ioc_buy":
            r = ioc_buy(engine, size)
        elif t == "fok_buy":
            r = fok_buy(engine, size)
        else:
            r = {"error": f"tipo desconocido: {t}"}
        results.append({"order_type": t, "size": size, **r})
    return results
```
</details>

---

## Ejercicio 10 — Modelo maker/taker fee *(Bonus / casa)*

En la mayoria de exchanges, los makers pagan menos fee que los takers (o incluso cobran rebate).

Supone el siguiente modelo de fees para BTCUSDT:
- **Taker fee:** 0.04% del valor notional
- **Maker fee:** 0.02% del valor notional (paga menos por aportar liquidez)

Calcula el coste total (precio + fee) de:
1. Comprar 1 BTC con market order (taker)
2. Comprar 1 BTC con limit order al mid (maker, si ejecuta)

¿El ahorro del spread compensa la diferencia de fees? ¿A partir de que tamano?

In [ ]:
TAKER_FEE = 0.0004  # 0.04%
MAKER_FEE = 0.0002  # 0.02%

sizes = [0.1, 0.5, 1.0, 2.0, 5.0]

print(f"{'Tamano':>8} | {'Market (taker)':>16} | {'Limit al mid (maker)':>20} | {'Ventaja limit':>14}")
print("-" * 70)

for size in sizes:
    e = MatchingEngine.from_row(row)
    mid = (e.best_bid() + e.best_ask()) / 2

    # TODO: calcular coste total para market y limit
    coste_market = 0  # avg_price * size * (1 + TAKER_FEE)
    coste_limit  = 0  # mid * size * (1 + MAKER_FEE)

    print(f"{size:>8.1f} | ${coste_market:>14,.2f} | ${coste_limit:>18,.2f} | ${coste_market - coste_limit:>12,.2f}")

<details>
<summary>Solucion ejercicio 10</summary>

```python
TAKER_FEE = 0.0004
MAKER_FEE = 0.0002

sizes = [0.1, 0.5, 1.0, 2.0, 5.0]

print(f"{'Tamano':>8} | {'Market (taker)':>16} | {'Limit al mid (maker)':>20} | {'Ventaja limit':>14}")
print("-" * 70)

for size in sizes:
    e = MatchingEngine.from_row(row)
    mid = (e.best_bid() + e.best_ask()) / 2
    r = market_buy(e, size)
    coste_market = r["avg_price"] * size * (1 + TAKER_FEE)
    coste_limit  = mid * size * (1 + MAKER_FEE)
    print(f"{size:>8.1f} | ${coste_market:>14,.2f} | ${coste_limit:>18,.2f} | ${coste_market - coste_limit:>12,.2f}")

# La ventaja de la limit crece con el tamano porque el spread y el slippage
# son costes proporcionales al tamano — y la limit los evita.
# Pero: la limit puede no ejecutar, y mientras espera el precio puede moverse en tu contra.
```
</details>

---

## Bien hecho

Has implementado un motor de matching completo desde cero usando los mismos datos de L4.

**Conceptos clave que has practicado:**
- Market orders caminan el libro nivel a nivel — el slippage crece con el tamano
- Limit orders esperan en el libro — precio garantizado, ejecucion incierta
- IOC = ejecuta lo que pueda, cancela el residuo; FOK = todo o nada
- El coste real de una orden incluye spread + slippage + fees

**Siguiente clase (L6):**
Ahora que sabes como se ejecutan las ordenes, construiremos un modelo de ciencia de datos para predecir la probabilidad de que una limit order ejecute, usando el imbalance del LOB como variable predictora.